# Retail lakehouse: bronze -> silver -> gold with PySpark and Delta Lake

A thin driver for the `spark/` package: all logic lives in the package and is unit-tested
there. Laid out like a Microsoft Fabric notebook: the first cell holds the parameters, so a
pipeline can override them. Step-by-step Fabric instructions: `docs/spark_on_fabric.md`.

The incremental batches are SIMULATED monthly replays of a static export, not change data capture.

In [ ]:
# Parameters
mode = "full"                    # "full" or "incremental"
data_dir = "data/raw"            # Fabric: "/lakehouse/default/Files/olist/raw" (or .../batches)
base_path = "lakehouse/full"     # Fabric: "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Tables"
repo_dir = ""                    # folder that contains spark/ and config/, if it is not the working directory

In [ ]:
import sys
if repo_dir:
    sys.path.insert(0, repo_dir)

# Notebooks already use the name `spark` for the SparkSession, so import the modules by name.
from spark import run as lakehouse
from spark.config import LakehouseConfig
from spark.session import get_spark

session = get_spark()            # returns the notebook's own session on Fabric
config = LakehouseConfig.create(base_path)
print(session.version, config.base_path)

In [ ]:
for source_dir, batch_id in lakehouse.batch_folders(session, data_dir, mode):
    summary = lakehouse.run_batch(session, config, source_dir, batch_id)
    lakehouse.print_summary(summary)

## Look at the result

In [ ]:
for table in ["gold_fact_orders", "gold_fact_order_items", "gold_fact_payments"]:
    df = session.read.format("delta").load(f"{config.base_path}/{table}")
    print(f"{table:<24}{df.count():>10,} rows")

(session.read.format("delta").load(config.table("gold", "dq_result"))
    .orderBy("checked_at_utc", ascending=False)
    .select("check_name", "status", "failed_rows").show(12, truncate=False))

In [ ]:
# Every MERGE is a version in the Delta transaction log, with its row counts
from delta.tables import DeltaTable
(DeltaTable.forPath(session, config.table("gold", "fact_orders")).history()
    .select("version", "operation", "operationMetrics.numTargetRowsInserted",
            "operationMetrics.numTargetRowsUpdated", "operationMetrics.numTargetRowsDeleted")
    .show(truncate=False))